In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import mne
import plotly.express as px

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'code'))
from compute_psd_epochs import load_all

In [ ]:
dataset = "ds003969"
tasks = ['think1', 'think2', 'med1breath', 'med2']

root = Path('/home/jackhoyle')
bids_root = root / 'mne_data' / dataset
processed_root = bids_root / 'processed'

In [ ]:
# process_inspect_loop.py stores e.g. "ICA excluded components: muscle=[11, 15], eog=[np.int64(2)]"
# in raw.info['description'], which can be read from each .fif without loading the data
def parse_ica_description(description):
    muscle, eog = re.search(r'muscle=\[(.*?)\], eog=\[(.*?)\]', description).groups()
    return [int(i) for i in re.findall(r'\d+', muscle)], [int(i) for i in re.findall(r'\d+', eog)]

rows = []
for fif in sorted(processed_root.glob('sub-*_task-*_processed_raw.fif')):
    subject, task = re.match(r'sub-(.+)_task-(.+)_processed_raw', fif.stem).groups()
    muscle, eog = parse_ica_description(mne.io.read_info(fif, verbose='error')['description'])
    rows.append(dict(
        subject=subject, task=task,
        n_muscle=len(muscle), n_eog=len(eog),
        n_total=len(set(muscle) | set(eog)),  # a component flagged as both is only excluded once
    ))
ica_df = pd.DataFrame(rows)
ica_df.head()

In [ ]:
# the subject/task combinations used in the analysis notebook (those with saved PSDs), and their groups
by_task, freqs, ch_names, subjects = load_all(
    bids_root / 'outputs/psd_epoch_mean',
    key='psd_mean'
)
used = pd.DataFrame([(s, t) for t, subs in subjects.items() for s in subs], columns=['subject', 'task'])

subject_info = pd.read_csv(bids_root / 'participants.tsv', sep='\t')
subject_info['subject'] = subject_info['participant_id'].str.removeprefix('sub-')

In [ ]:
df = (
    used
    .merge(ica_df, on=['subject', 'task'], how='left', validate='one_to_one')
    .merge(subject_info[['subject', 'group']], on='subject', how='left', validate='many_to_one')
)
assert df[['n_muscle', 'n_eog', 'group']].notna().all().all()
df.groupby('group').subject.nunique()

In [ ]:
# mean number of components excluded per group and task
df.groupby(['task', 'group'])[['n_muscle', 'n_eog', 'n_total']].mean().round(2)

In [ ]:
long = df.melt(
    id_vars=['subject', 'task', 'group'], value_vars=['n_muscle', 'n_eog'],
    var_name='component_type', value_name='n_components',
)
long['component_type'] = long['component_type'].map({'n_muscle': 'muscle', 'n_eog': 'eye (EOG)'})

px.box(
    long, x='group', y='n_components', color='group', points='all',
    facet_col='task', facet_row='component_type',
    category_orders={'task': tasks, 'group': sorted(df.group.unique())},
    hover_data=['subject'],
    labels={'n_components': 'ICA components excluded'},
    title='ICA components excluded by group',
    height=700,
)

In [ ]:
# mean ± SE of the number removed, per group (all tasks pooled)
summary = long.groupby(['component_type', 'group']).n_components.agg(['mean', 'sem']).reset_index()
px.bar(
    summary, x='group', y='mean', error_y='sem', color='component_type', barmode='group',
    labels={'mean': 'mean ICA components excluded', 'component_type': ''},
    title='Mean components excluded by group (all tasks pooled)',
)